> 📄 English: [web_scraping_agent.ipynb](web_scraping_agent.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--web-scraping-agent)

# 使用 LangGraph 与 Bright Data + MCP 构建智能网页抓取 Agent

## 概述

本综合教程演示了如何构建一个生产就绪的网页抓取 Agent，它将 LangGraph 的 ReAct（Reasoning and Acting，推理与行动）框架与 Bright Data 先进的抓取基础设施相结合。最终系统能够智能地浏览网站、提取结构化数据，并在极少人工干预的情况下执行复杂的研究工作流。

你将构建的 Agent 相比传统网页抓取方法是一大进步。你不再需要为每个网站编写定制化的抓取器，而是创建一个能够推理不同抓取策略、选择合适工具并自动适应各种网页结构的智能系统。

## 学习目标

完成本教程后，你将理解如何：

- 借助外部工具集成实现 LangGraph ReAct Agent，用于自主决策
- 配置 Bright Data 的 Model Context Protocol（MCP）服务器，实现企业级网页抓取
- 设计能够根据目标网站动态选择最优抓取策略的智能 Agent
- 从主流平台（包括电商网站、社交媒体和新闻来源）提取结构化数据
- 实现浏览器自动化工作流，以处理复杂的用户交互
- 构建综合研究流水线，综合来自多个来源的信息

## 架构概览

系统架构由三个协同工作的核心组件组成：

```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────────┐
│   User Query    │───▶│   ReAct Agent    │───▶│   Bright Data MCP   │
│                 │    │   (LangGraph)    │    │      Server         │
└─────────────────┘    │                  │    │                     │
                       │ ┌──────────────┐ │    │ ┌─────────────────┐ │
                       │ │ Reasoning    │ │    │ │ Search Engines  │ │
                       │ │ Engine       │ │    │ │ Web Scrapers    │ │
                       │ └──────────────┘ │    │ │ Platform APIs   │ │
                       │ ┌──────────────┐ │    │ │ Browser Tools   │ │
                       │ │ Tool         │ │    │ └─────────────────┘ │
                       │ │ Selection    │ │    └─────────────────────┘
                       │ └──────────────┘ │
                       └──────────────────┘
```

ReAct Agent 充当智能协调者，分析用户请求并从 Bright Data 全面的网页抓取能力套件中选择合适的工具。这种设计既保证了处理多样化网页抓取场景的灵活性，也保证了可靠性。


## 前置条件与环境配置

在开始实现之前，你需要注册账号并配置开发环境。本节将引导你完成必要的配置步骤。

### 所需账号与 API Key

**Bright Data 账号配置**

1. 在[此链接](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--web-scraping-agent&click=brightdata&target=https%3A%2F%2Fbrightdata.com%2F%3Fhs_signup%3D1%26utm_source%3Dbrand%26utm_campaign%3Dbrnd-mkt_github_nirdiamant_logo&text=This%20link)创建一个 Bright Data 账号。新账号每月可免费获得 5,000 次 unlocker 请求，为开发和测试提供充足资源。

    ![截图：注册页面](assets/Signup.png)

2. 前往你的[账户设置](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-brightdata--web-scraping-agent&click=brightdata-setting&target=https%3A%2F%2Fbrightdata.com%2Fcp%2Fsetting&text=account%20settings)并找到你的 API key。该凭据将用于让 Agent 通过 Bright Data 的基础设施进行身份认证。

    ![截图：设置页面](assets/Settings.png)

**语言模型访问**

3. 注册一个 OpenRouter 账号，以获取语言模型的 API 访问权限。虽然本教程通过 OpenRouter 使用 Gemini 以获得最佳性能和成本效益，但该架构支持任何兼容的语言模型。

**开发环境**

4. 确保你的开发环境包含 Python 3.8 或更高版本，以及 pip 包管理能力。


## 依赖安装与配置

以下部分安装所需的 Python 包，并为我们的网页抓取 Agent 建立基础导入。每个依赖在整体架构中都扮演着特定角色。


### API Key 配置

该单元格为你的 API 凭据创建环境变量。执行前请将占位值替换为你的实际 API key。


In [ ]:
# To export your API key into a .env file, run the following cell (replace with your actual keys):
!echo "BRIGHT_DATA_API_TOKEN=<your-brightdataa-api-key>" >> .env
!echo "OPENROUTER_API_KEY=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" >> .env

### 包安装

该单元格安装网页抓取 Agent 所需的核心依赖。这些包包括用于 Agent 编排的 LangGraph、用于语言模型交互的 OpenAI 客户端、用于 Bright Data 集成的 MCP 客户端，以及配套工具。


In [ ]:
# Install required packages
%pip install langgraph langchain-openai mcp-use python-dotenv asyncio --quiet
%load_ext autoreload
%autoreload 2

### 核心导入与环境变量加载

该单元格导入必要的库，并从配置文件加载环境变量。导入内容包括异步处理能力、语言模型客户端、Agent 框架以及 MCP 集成组件。


In [ ]:
# Import all necessary libraries
import asyncio
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from mcp_use.client import MCPClient
from mcp_use.adapters.langchain_adapter import LangChainAdapter
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

### 环境验证

在继续配置 Agent 之前，验证所有必需的环境变量都已正确加载至关重要。这一验证步骤可以防止运行时错误，确保顺利运行。


In [ ]:
# Verify environment setup
print("✅ Environment setup complete!")
print(f"OpenRouter API Key loaded: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No'}")

## 理解 ReAct Agent 框架

ReAct（Reasoning and Acting，推理与行动）框架代表了自主 Agent 设计的一大进步。与传统的基于规则的系统或简单的思维链方法不同，ReAct Agent 将推理能力与动作执行整合在一个统一的框架中。

### ReAct 架构的核心原则

ReAct Agent 运行在一个将深思熟虑的推理与交互式工具使用相结合的基本原则之上。这种方法使 Agent 能够：

**动态推理**：Agent 分析每种情况，并针对具体任务需求制定推理策略。它不会遵循预设脚本，而是评估可用信息并确定最有效的方法。

**工具选择**：基于推理结果，Agent 从可用工具库中选择合适的工具。这一选择过程会考虑数据需求、网站特征以及期望的输出格式等因素。

**迭代优化**：Agent 可以按顺序执行多个动作，利用前一个动作的结果来指导后续决策。这种能力对于需要多步工作流的复杂网页抓取场景尤为宝贵。

### 在网页抓取中的应用

在网页抓取场景下，ReAct Agent 表现出色，因为它们能够根据目标网站的特征调整方法。例如，遇到电商网站时，Agent 可能会识别出对结构化产品数据提取的需求。反之，在分析新闻文章时，它可能会优先采用内容提取和摘要技术。

这种适应性消除了为不同网站维护独立抓取脚本的需要，显著降低了开发开销，同时提高了可靠性和可维护性。


## Bright Data MCP 集成

Model Context Protocol（MCP）代表了一种将外部服务与语言模型应用集成的标准化方法。Bright Data 的 MCP 服务器通过统一接口提供一整套网页抓取工具。

### MCP 架构的优势

相比直接集成 API，MCP 集成具有多项优势：

**标准化接口**：所有 Bright Data 工具都通过一致的函数签名访问，简化了 Agent 开发并降低了集成复杂度。

**自动工具发现**：MCP 服务器动态暴露可用工具，使 Agent 无需修改代码即可发现和利用新能力。

**错误处理**：内置的错误处理和重试机制，在处理网络问题或服务临时不可用时提高了可靠性。

**性能优化**：MCP 服务器包含缓存和请求优化功能，提升了整体系统性能。


### Bright Data MCP 服务器配置

该函数建立与 Bright Data MCP 服务器的连接，并将可用工具转换为 LangChain 兼容的格式。配置过程包括设置服务器连接参数，并使用你的 API 凭据进行身份认证。


In [ ]:
async def setup_bright_data_tools():
    """
    Configure Bright Data MCP client and create LangChain-compatible tools
    """
    
    # Configure the MCP server connection
    bright_data_config = {
        "mcpServers": {
                    "Bright Data": {
            "command": "npx",
            "args": ["@brightdata/mcp"],
            "env": {
                "API_TOKEN": os.getenv("BRIGHT_DATA_API_TOKEN"),
            }
            }
        }
    }
    
    # Create MCP client and adapter
    client = MCPClient.from_dict(bright_data_config)
    adapter = LangChainAdapter()
    
    # Convert MCP tools to LangChain-compatible format
    tools = await adapter.create_tools(client)
    
    print(f"✅ Connected to Bright Data MCP server")
    print(f"📊 Available tools: {len(tools)}")
    
    return tools

# Test the connection
tools = await setup_bright_data_tools()

### 可用工具类别

Bright Data MCP 服务器提供多类网页抓取工具的访问，每类工具都针对特定用例进行了优化：

**搜索引擎集成**：全面访问包括 Google、Bing 和 Yandex 在内的主要搜索引擎，使 Agent 能够根据用户查询发现相关网页内容。

**通用网页抓取**：通用的抓取能力，可以以 Markdown 和 HTML 等多种格式从任意网站提取内容，并内置爬虫检测绕过机制。

**平台专用提取器**：针对 Amazon、LinkedIn、Instagram、Facebook、X（Twitter）、TikTok、YouTube、Reddit 和 Zillow 等主流平台的专用工具。这些提取器理解各平台的特定数据结构，比通用抓取器更高效地提取信息。

**浏览器自动化**：模拟用户交互的高级能力，包括导航、点击、输入和截图。这些工具对于需要复杂用户交互或执行 JavaScript 的网站至关重要。

可用工具的多样性确保 Agent 能够以最优的效率和可靠性处理几乎任何网页抓取场景。


## 语言模型与 Agent 配置

下一阶段是配置语言模型，并使用全面的系统提示词创建 ReAct Agent。系统提示词在定义 Agent 的行为、推理模式和工具选择策略方面起着至关重要的作用。


### ReAct Agent 初始化

该函数将语言模型与 Bright Data 工具相结合，创建完整的网页抓取 Agent。系统提示词经过精心设计，用于引导 Agent 的决策过程，确保最优的工具利用。


In [ ]:
import datetime

async def create_web_scraper_agent():
    """
    Create a ReAct agent configured for intelligent web scraping
    """
    
    # Get the tools from Bright Data first
    tools = await setup_bright_data_tools()
    
    # Get current date
    current_date = datetime.datetime.now().strftime("%B %d, %Y")
    
    # Initialize the language model
    llm = ChatOpenAI(
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        openai_api_base="https://openrouter.ai/api/v1",
        model_name="google/gemini-2.5-flash-lite-preview-06-17",  # Fast and capable model for reasoning
        temperature=0.1  # Low temperature for consistent, focused responses
    )
    
    # Define comprehensive system prompt for the agent with dynamic date and tool count
    system_prompt = f"""You are a web data extraction agent. Today's date is {current_date}.

You have {len(tools)} specialized tools for web scraping and data extraction. When users request web data or current information, you MUST use these tools - do not rely on your training data.

Available capabilities:
- Search engines (Google/Bing/Yandex)
- Universal web scraping (any website)
- Platform extractors (Amazon, LinkedIn, Instagram, Facebook, X, TikTok, YouTube, Reddit, etc.)
- Browser automation

Process:
1. Identify data need
2. Select appropriate tool
3. Execute extraction
4. Return structured results

Always use tools for current/live data requests."""
    
    # Create the ReAct agent
    agent = create_react_agent(
        model=llm,
        tools=tools,
        prompt=system_prompt
    )
    
    print("🤖 ReAct Web Scraper Agent created successfully!")
    return agent

# Create the agent
agent = await create_web_scraper_agent()

### 系统提示词设计原则

系统提示词是 Agent 行为的基础，融入了几个关键的设计原则：

**清晰的角色定义**：提示词确立了 Agent 作为网页数据提取专家的身份，为其主要功能提供清晰上下文。

**工具意识**：通过明确说明可用工具的数量和类型，提示词确保 Agent 了解自身能力，并优先使用工具而非依赖训练数据。

**流程框架**：提示词中概述的四步流程为处理用户请求提供了结构化方法，确保回复一致且有条理。

**时间上下文**：包含当前日期有助于 Agent 理解请求的时间背景，并在相关时优先考虑最新信息。

**强制使用工具**：针对当前数据请求必须使用工具的指令，防止 Agent 提供来自训练数据的过时信息。


## 基础搜索功能测试

Agent 配置完成后，我们现在可以测试其基础能力。第一个测试演示了 Agent 搜索当前信息并从多个来源综合结果的能力。

### 理解 Agent 的决策过程

面对搜索查询时，Agent 遵循系统化的决策过程。它首先分析查询以理解信息需求，然后根据查询特征选择最合适的搜索工具。接着 Agent 执行搜索、分析结果，并呈现结构化的摘要。


### 基础搜索查询执行

该测试演示了 Agent 搜索当前信息并将多个搜索结果处理为连贯摘要的能力。Agent 会自动选择合适的搜索引擎，并以结构化格式呈现发现。


In [ ]:
async def test_basic_search():
    """
    Test the agent's ability to search for current information
    """
    
    print("Testing Basic Search Functionality...")
    print("="*50)
    
    # Simple search query
    search_result = await agent.ainvoke({
        "messages": [("human", "Give me the latest AI news from this week, Include full URLs to source.")],
    })
    
    print("\n🔍 Search Results:")
    print(search_result["messages"][-1].content)
    
    return search_result

# Execute the test
basic_search_result = await test_basic_search()

### 搜索结果分析

搜索功能展示了 ReAct Agent 的几项关键能力：

**查询理解**：Agent 正确解读了对近期 AI 新闻的请求，并理解了时间要求（"本周"）。

**工具选择**：根据查询类型，Agent 从可用选项中选择合适的搜索引擎工具。

**结果处理**：Agent 聚合来自多个来源的结果，并以结构化、易读的格式呈现，附带来源 URL。

**内容优先级**：结果展示了 Agent 识别并优先处理来自权威来源的相关、最新信息的能力。


## 高级平台专用数据提取

除了一般的网页搜索，Agent 真正的威力在于从特定平台提取结构化数据的能力。每个平台在数据结构、访问方式和内容组织方面都带来独特挑战。

### 电商平台分析

像 Amazon 这样的电商平台包含丰富的结构化数据，包括产品规格、定价、评论和库存信息。Agent 的平台专用工具可以驾驭这些复杂的数据结构，高效地提取相关信息。

分析电商数据时，Agent 会考虑多个因素，包括产品特性、价格趋势、客户反馈，以及相似产品之间的对比分析。这种全面的方法为用户提供可用于购买决策或市场研究的可执行洞察。


### 电商数据提取与分析

该测试演示了 Agent 在电商平台上研究和对比产品的能力。Agent 会搜索产品、提取结构化数据，并提供包含价格和功能信息的对比分析。


In [ ]:
async def test_ecommerce_scraping():
    """
    Test structured data extraction from e-commerce platforms
    """
    
    print("Testing E-commerce Data Extraction...")
    print("="*50)
    
    # Ask the agent to find and analyze a product
    ecommerce_result = await agent.ainvoke({
        "messages": [("human", "Find information about the top-rated wireless headphones on Amazon and compare their features and prices")]
    })
    
    print("\n🛒 E-commerce Analysis:")
    print(ecommerce_result["messages"][-1].content)
    
    return ecommerce_result

# Execute the test
ecommerce_result = await test_ecommerce_scraping()

### 电商分析能力

电商分析展示了多项复杂能力：

**产品发现**：Agent 可以在特定品类内搜索产品，并根据客户评论和评分识别高评分选项。

**功能提取**：详细的产品规格、功能和技术细节被系统地提取并整理，便于对比。

**价格分析**：采集当前价格信息，并在可用时与历史价格趋势一并呈现。

**对比框架**：Agent 通过突出产品之间的关键差异来组织分析，以辅助决策。

**可执行建议**：Agent 不只是呈现数据，还会根据不同的使用场景和预算考量提供指导。


## 社交媒体内容分析

社交媒体平台因其动态特性、复杂的用户界面和多样化的内容格式，为数据提取带来了独特挑战。Agent 的社交媒体能力使其能够分析各平台上的讨论、情绪和热门话题。

### Reddit 讨论分析

Reddit 的社区（subreddit）结构和串式讨论结构，为了解公众舆论和特定主题的专家见解提供了丰富机会。Agent 可以浏览 Reddit 界面、提取相关讨论，并分析内容以获取关键主题和洞察。


### Reddit 内容提取与分析

该测试演示了 Agent 搜索和分析社交媒体内容的能力。Agent 会搜索相关 Reddit 讨论，并提取关键对话话题和社区洞察。


In [ ]:
async def test_social_media_simple():
    """
    Test Reddit extraction with a specific approach
    """
    print("Testing Reddit Extraction...")
    print("="*50)
    
    result = await agent.ainvoke({
        "messages": [("human", "Search for 'electric vehicles reddit' and then scrape one of the Reddit discussion pages you find. Show me what people are discussing.")]
    })
    
    print("\n📱 Reddit Analysis:")
    print(result["messages"][-1].content)
    return result

# Test this version
social_simple = await test_social_media_simple()

### 社交媒体分析洞察

社交媒体分析展示了多项关键能力：

**平台专用导航**：Agent 理解 Reddit 的结构，可以导航到特定的 subreddit 和讨论串。

**内容聚合**：分析多条帖子和评论，以识别共同主题和讨论话题。

**情绪分析**：Agent 可以解读讨论的语气和情绪，识别对话题的正面、负面或中立态度。

**趋势识别**：通过分析多个讨论，Agent 可以识别社区内的新兴趋势和热门话题。

**上下文理解**：在解读内容时，Agent 会保持对特定社区文化和讨论规范的上下文理解。


## 复杂的多步研究工作流

Agent 最复杂的能力在于执行需要多个工具、多个推理步骤和信息综合的复杂研究。这些工作流展示了 ReAct 框架在处理现实研究场景中的全部潜力。

### 研究工作流架构

复杂研究工作流涉及多个相互关联的阶段：

**任务分解**：Agent 分析复杂查询，将其分解为可单独处理的可管理子任务。

**工具编排**：按顺序使用多个工具，每个工具的输出指导后续工具的选择和配置。

**信息综合**：分析、比较来自各种来源的数据，并将其综合为回应原始研究问题的连贯洞察。

**质量验证**：Agent 评估所收集信息的质量和相关性，识别需要进一步研究的缺口。


### 多步研究执行

该测试用一个需要多个信息源和分析步骤的复杂研究任务来挑战 Agent。Agent 必须展示其规划研究步骤、系统地执行步骤并将发现综合为可执行洞察的能力。


In [ ]:
async def test_complex_research():
    """
    Test the agent's ability to conduct multi-step research
    """
    
    print("Testing Complex Multi-Step Research...")
    print("="*50)
    
    # Complex research query
    research_result = await agent.ainvoke({
        "messages": [("human", """
        I need to research the current state of the renewable energy market. Please:
        1. Find recent news about renewable energy developments
        2. Look up major renewable energy companies and their stock performance
        3. Analyze social media sentiment about renewable energy
        4. Provide a comprehensive market overview with key insights
        """)]
    })
    
    print("\n🔬 Complex Research Results:")
    print(research_result["messages"][-1].content)
    
    return research_result

# Execute the test
research_result = await test_complex_research()

### 研究方法论与规划

Agent 对复杂研究请求的回应展示了成熟的规划能力：

**需求分析**：Agent 评估研究请求，并识别出补充说明可以提高研究质量的领域。

**范围界定**：Agent 不急于假设，而是寻求澄清，以确保研究工作聚焦且相关。

**工具映射**：Agent 理解哪些工具适合不同类型的信息收集，并概述其方法。

**质量保证**：通过请求具体参数，Agent 确保最终的研究输出符合专业标准和用户期望。

这种规划方法反映了专业研究方法论中的最佳实践，展示了 Agent 处理企业级研究需求的能力。


## 综合研究助手实现

教程的最后一部分是创建一个综合研究助手函数，展示 Agent 所有能力的完整集成。该函数可作为构建生产就绪研究工作流的模板。

### 研究助手架构

研究助手函数融入了几个使其适合生产使用的设计原则：

**参数化配置**：用户可以指定研究范围、来源数量限制等参数来控制研究过程。

**结构化输出**：研究结果以一致的格式组织，便于进一步分析或报告。

**进度跟踪**：函数提供研究进度的反馈，帮助用户理解所执行工作的复杂性和范围。

**质量控制**：内置机制确保研究聚焦于高质量、相关的来源，而非简单地最大化收集信息的数量。


### 研究助手函数实现

这个综合函数演示了如何组织复杂的研究查询，并为 Agent 配置最优性能。该函数包含研究范围的参数化，并提供结构化的输出格式。


In [ ]:
async def research_assistant(query: str, max_sources: int = 5):
    """
    A comprehensive research assistant using our web scraping agent
    
    Args:
        query (str): The research question or topic
        max_sources (int): Maximum number of sources to analyze
    """
    
    print(f"🔍 Starting research on: {query}")
    print("="*60)
    
    # Enhanced research prompt
    research_prompt = f"""
    Please conduct comprehensive research on: "{query}"
    
    Your research should include:
    1. Current news and developments (last 30 days)
    2. Expert opinions and analysis
    3. Statistical data and trends
    4. Social media sentiment (if relevant)
    5. Key players and companies involved
    
    Use multiple sources and provide a well-structured summary with:
    - Executive summary
    - Key findings
    - Supporting data
    - Sources used
    
    Limit your research to {max_sources} high-quality sources.
    """
    
    result = await agent.ainvoke({
        "messages": [("human", research_prompt)]
    })
    
    print(f"\n📊 Research Complete!")
    print(result["messages"][-1].content)
    
    return result

# Test the research assistant
research_result = await research_assistant("Impact of artificial intelligence on job markets in 2025",5)

### 研究助手的好处与应用

综合研究助手函数展示了几个关键好处：

**可扩展的研究框架**：参数化方法使同一个函数能够处理不同范围和复杂度的研究项目。

**一致的输出结构**：结构化的提示词确保研究结果遵循可预测的格式，便于与其他系统或流程集成。

**质量优先**：通过限制来源数量并强调质量，函数在研究覆盖面上优先追求深度而非广度。

**专业标准**：研究方法论融入了专业研究机构的最佳实践，确保输出满足企业要求。

**可定制能力**：该函数可以轻松修改，以应对特定行业需求或研究方法论。
